<a href="https://colab.research.google.com/github/Linux-Server/Transformers/blob/main/Google-Bert-Base-Cased/02_PEFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


check_point = "facebook/opt-350m"


pipe = pipeline("text-generation", model=check_point)





In [ ]:
pipe("I love you", max_new_tokens= 60)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(check_point)

In [ ]:
from torchinfo import summary

summary(model)

In [ ]:
# create peft config
from peft import LoraConfig, TaskType

peft_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05)

In [ ]:
from peft import get_peft_model

lora_model = get_peft_model(model, peft_config)

In [ ]:
lora_model.print_trainable_parameters()

In [ ]:
## Fine the model for intruction tuning with
from datasets import load_dataset


raw = load_dataset("vicgalle/alpaca-gpt4")

raw = raw["train"].select(range(10000))
raw

In [ ]:
def format_alpaca(example):
    inst, inp, out = example["instruction"].strip(), example["input"].strip(), example["output"].strip()
    if inp:
        prompt = f"<|user|>\n{inst}\n\n### Input:\n{inp}\n<|assistant|>\n"
    else:
        prompt = f"<|user|>\n{inst}\n<|assistant|>\n"
    full = prompt + out + "</s>"
    return {"text": full}





In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-350m", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    prompt = format_alpaca(example)["text"]
    tokens = tokenizer(prompt, truncation=True, max_length=1024, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tok_ds = raw.map(tokenize, remove_columns=raw.column_names, num_proc=4)



In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

base = AutoModelForCausalLM.from_pretrained(
    "facebook/opt-350m",
    torch_dtype="float16",
    device_map="auto"
)

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],   # OPT projection layers
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()


In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./opt350m-alpaca-chat",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,     # effective 16
    num_train_epochs=3,
    fp16=True,
    learning_rate=2e-4,                # LoRA can take a higher LR
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=1,
    optim="adamw_torch"
)

trainer = Trainer(model=model, args=args, train_dataset=tok_ds)


In [ ]:
trainer.train()

In [ ]:
def chat(instruction, input_text=None, max_new_tokens=256):
    if input_text:
        prompt = f"<|user|>\n{instruction}\n\n### Input:\n{input_text}\n<|assistant|>\n"
    else:
        prompt = f"<|user|>\n{instruction}\n<|assistant|>\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("<|assistant|>\n")[-1].strip()



In [ ]:
chat("Who are you?")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
model.save_pretrained("./opt350m-alpaca-10k-instruction-tune-3")

In [ ]:
model.push_to_hub("opt350m-alpaca-10k-instruction-tune-3")